# Computational Set: Data Analysis D — Nonlinear Curve Fitting

### Bond Length of a Diatomic from a Morse Potential

**Learning objectives**

- Fit data to a **general nonlinear function** (one that cannot be turned into a polynomial).
- Obtain best-fit parameters **and** their uncertainties from a single tool.
- Connect fitted parameters to molecular properties (bond length, well depth, curvature).

---

#### Background

When a model is exponential, logarithmic, or otherwise not a polynomial, the regression tricks of
A–C don't apply. In Excel you used **Solver** to minimize the sum of squared residuals ($\chi^2$),
then a special macro to get the parameter uncertainties. Python does both at once with
`scipy.optimize.curve_fit`, which returns the best-fit parameters and their **covariance matrix** —
the square roots of its diagonal are the uncertainties, replacing the macro entirely.

Here we fit the electronic energy of CO vs internuclear distance to the **Morse potential**:

$$
V(r) = D_e\bigl(1 - e^{-\beta (r - r_e)}\bigr)^2, \tag{8}
$$

where $D_e$ is the well depth, $r_e$ the equilibrium bond length, and $\beta$ sets the curvature.


> **New syntax in this set.**
> - **Defining a function:** `def morse(r, De, beta, re): return ...` packages the model so the
>   fitter can call it. The first argument is the independent variable; the rest are parameters to fit.
> - `np.exp(x)` is $e^x$, element by element.
> - **`curve_fit(model, x, y, p0=[...])`** returns two things: `popt` (the best-fit parameter
>   array) and `pcov` (their covariance matrix). It needs **initial guesses** `p0` — nonlinear
>   fits are iterative and can fail from a poor start, so we estimate the parameters from the plot first.
>
> Fuller refresher: ESCIP [“What is Python?”](https://escip.io/notebooks/python/python-basics-fixed.html).

## The data

A Gaussian DFT scan produced the electronic energy of CO at a series of bond lengths. Energies are
in **kcal/mol** (`V_kcal`, reported relative to the first geometry, as Gaussian does) and distances
in **ångström** (`r`).

💾 **Run the cell below to load the data — you don't need to edit it.**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import FileUpload, Dropdown, VBox
from IPython.display import display
import io
import pandas as pd

uploader = FileUpload(accept='.csv', multiple=False, description='Upload CSV')
display(VBox([uploader]))

# 📥 Run the cell below to load your data - **you don't need to edit it.**

In [ ]:
# Run this cell AFTER you've uploaded your file above.

if len(uploader.value) == 0:
    raise ValueError("No file uploaded yet — click 'Upload CSV' above, choose your file, then re-run this cell.")

# ipywidgets FileUpload.value is a tuple of dicts (v8+) or a dict keyed by filename (v7)
uploaded = uploader.value[0] if isinstance(uploader.value, tuple) else list(uploader.value.values())[0]
raw = pd.read_csv(io.BytesIO(bytes(uploaded['content'])), header=None)

# --- First two columns: r (Angstroms),V (kcal / mol ---
data = raw.iloc[1:, 0:2].reset_index(drop=True).astype(float)
data.columns = raw.iloc[0, 0:2].tolist()

r  = data.iloc[:, 0].values
V  = data.iloc[:, 1].values



print(f"{len(r)} data points loaded from file.")
print(f"Bond length = {r} in Angstroms,  V = {V} kcal / mol")


## Step 1 — Prepare the energies

Two small conversions before fitting:

- convert the energies from kcal/mol to **kJ/mol** (multiply by `KCAL_TO_KJ`);
- the Morse function is **zero at its minimum**, but the Gaussian energies are referenced to the
  first geometry, so **shift** the data down by its minimum value to make the lowest energy zero.

👉 **Your task:** complete the two expressions.

In [ ]:
# Conversion from kcal / mol to kj/mol
KCAL_TO_KJ = 4.184

# convert kcal/mol to kJ/mol
V_kJ =               # <-- replace with your expression


# shift so the minimum energy is zero
Vdata =               # <-- subtract the minimum of V_kJ from V_kJ   (hint: np.min)

print("Vdata (kJ/mol):", np.round(Vdata, 1))

## Step 2 — Plot the data and estimate starting values

Plot the prepared energy `Vdata` (vertical) against `r` (horizontal) as open markers. Then read
rough starting values off the plot, because `curve_fit` needs an initial guess:

- $r_e$ ≈ the $r$ at the minimum,
- $D_e$ ≈ the energy where the curve levels off at large $r$,
- $\beta$ ≈ 2 Å⁻¹ is a reasonable generic start.

👉 **Your task:** supply the `ax.plot(...)` call for the data points.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

# Plot r (x) vs Vdata (y) as open circles, no connecting line.
ax.plot(                   )   # <-- pass r, Vdata, and the format string

ax.set_xlabel(r'$r$ (Å)')
ax.set_ylabel(r'$V$ (kJ/mol)')
ax.tick_params(direction='in')
plt.show()

## Step 3 — Define the model and fit it

First write the Morse function of Eq. (8), then call `curve_fit` with your starting guesses.

👉 **Your task:** complete the `return` line of `morse`, set the initial guesses `p0`, and call `curve_fit`.

In [ ]:
# Morse potential, Eq. (8). Inputs: distance r and parameters De, beta, re.
def morse(r, De, beta, re):
    return                          # <-- De * (1 - np.exp(-beta * (r - re)))**2

# initial guesses [De, beta, re], read from the plot (De ~ plateau, beta ~ 2, re ~ minimum)
p0 = [                ]             # <-- e.g. [1000, 2.0, 1.13]

# fit: returns best-fit parameters and their covariance matrix
popt, pcov =                        # <-- curve_fit(morse, r, Vdata, p0=p0)

De, beta, re = popt
S_De, S_beta, S_re = np.sqrt(np.diag(pcov))

print(f"De   = {De:.1f}  (S = {S_De:.1f}) kJ/mol")
print(f"beta = {beta:.3f} (S = {S_beta:.3f}) 1/Å")
print(f"re   = {re:.4f} (S = {S_re:.4f}) Å")

## Step 4 — Overlay the fitted curve

Evaluate the Morse function on a smooth, dense set of $r$ values using your fitted parameters, and
plot it over the data. `np.linspace(a, b, n)` makes `n` evenly spaced points from `a` to `b`.

👉 **Your task:** evaluate the model on the smooth grid.

In [ ]:
r_smooth = np.linspace(r.min(), r.max(), 200)

# evaluate the fitted Morse curve on r_smooth using the fitted parameters
V_fit =                  # <-- morse(r_smooth, De, beta, re)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(r, Vdata, 'o', mfc='none', ms=6, label='data')
ax.plot(r_smooth, V_fit, '-', lw=1, label='Morse fit')
ax.set_xlabel(r'$r$ (Å)')
ax.set_ylabel(r'$V$ (kJ/mol)')
ax.tick_params(direction='in'); ax.legend()
plt.show()

## Step 5 — Report the parameters with correct significant figures

Report all three Morse parameters with the lab's sig-fig rules using the `report` helper.

👉 **Your task:** call `report` for $D_e$, $\beta$, and $r_e$.

In [ ]:
from math import log10, floor

# Format 'value +/- uncertainty unit' following the lab's two sig-fig rules.
def report(value, unc, unit=""):
    if unc == 0:
        return f"{value} {unit}"
    exp = floor(log10(abs(unc)))
    lead = int(abs(unc) / 10**exp)
    sig = 2 if lead in (1, 2) else 1
    dp = -(exp - (sig - 1))
    if dp >= 0:
        return f"{value:.{dp}f} ± {unc:.{dp}f} {unit}"
    f = 10**(-dp)
    return f"{round(value/f)*f:g} ± {round(unc/f)*f:g} {unit}"

print("De   =", report(            ))   # <-- De, S_De, "kJ/mol"
print("beta =", report(            ))   # <-- beta, S_beta, "1/Å"
print("re   =", report(            ))   # <-- re, S_re, "Å"


## Discussion

Answer briefly below (as Markdown):

1. How well does the Morse curve match the points across the whole range? Where is the agreement worst, and why might that be?
2. Compare your fitted $r_e$ to CO's accepted bond length (~1.128 Å). Is it consistent within the uncertainty?
3. Re-run the fit with a deliberately poor `p0` (e.g. `[10, 0.1, 3.0]`). What happens, and what does that teach you about nonlinear fitting?

*Your answers here.*